# Module 4 – Interest Rate Swap Trade Representation

In this module we build the object model required to represent a vanilla
Interest Rate Swap (IRS).

The objective is not pricing.

Instead, we construct clean domain objects that accurately describe an IRS,
which later modules will use for schedule generation, pricing, sensitivities,
and risk calculations.

Classes introduced

- Trade
- Leg
- FixedLeg
- FloatingLeg
- InterestRateSwap
- Cashflow

Before pricing a financial instrument, we first need a way to represent it.

A pricing engine should never receive random variables like

- Notional
- Rate
- Dates

Instead it should receive a complete object describing the trade.

This separation keeps the system modular and closely resembles enterprise
risk systems used in investment banks.

```
Trade
        │
        ▼
InterestRateSwap
      /          \
FixedLeg      FloatingLeg
        │
        ▼
     Cashflows
```

In [1]:
from datetime import date

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PaymentFrequency,
    PayReceive,
)

from src.products.trade import Trade
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap

In [2]:
trade = Trade(
    trade_id="IRS001",
    counterparty="Goldman Sachs",
    currency="USD",
    effective_date=date(2026,1,1),
    maturity_date=date(2031,1,1),
)

fixed_leg = FixedLeg(
    notional=100_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
)

floating_leg = FloatingLeg(
    notional=100_000_000,
    floating_index=FloatingIndex.SOFR,
    spread=0.001,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

swap

InterestRateSwap(trade=Trade(trade_id='IRS001', counterparty='Goldman Sachs', currency='USD', effective_date=datetime.date(2026, 1, 1), maturity_date=datetime.date(2031, 1, 1)), fixed_leg=FixedLeg(notional=100000000, payment_frequency=<PaymentFrequency.ANNUAL: 1>, day_count=<DayCountConvention.ACT_365: 'ACT/365'>, pay_receive=<PayReceive.PAY: 'PAY'>, fixed_rate=0.05), floating_leg=FloatingLeg(notional=100000000, payment_frequency=<PaymentFrequency.ANNUAL: 1>, day_count=<DayCountConvention.ACT_365: 'ACT/365'>, pay_receive=<PayReceive.RECEIVE: 'RECEIVE'>, floating_index=<FloatingIndex.SOFR: 'SOFR'>, spread=0.001))

In [3]:
print("Trade ID      :", swap.trade.trade_id)
print("Counterparty  :", swap.trade.counterparty)
print("Currency      :", swap.trade.currency)
print("Fixed Rate    :", swap.fixed_leg.fixed_rate)
print("Floating Index:", swap.floating_leg.floating_index.value)
print("Spread        :", swap.floating_leg.spread)
print("Notional      :", f"{swap.fixed_leg.notional:,.0f}")

Trade ID      : IRS001
Counterparty  : Goldman Sachs
Currency      : USD
Fixed Rate    : 0.05
Floating Index: SOFR
Spread        : 0.001
Notional      : 100,000,000


In [4]:
from src.products.cashflow import Cashflow

cf = Cashflow(
    payment_date=date(2027,1,1),
    accrual_start=date(2026,1,1),
    accrual_end=date(2027,1,1),
    year_fraction=1.0,
    amount=5_000_000,
)

cf

Cashflow(payment_date=datetime.date(2027, 1, 1), accrual_start=datetime.date(2026, 1, 1), accrual_end=datetime.date(2027, 1, 1), year_fraction=1.0, amount=5000000)

In this module we created a complete representation of a vanilla Interest Rate
Swap.

The product model is intentionally independent from pricing logic.

This separation allows future modules to generate payment schedules,
construct cashflows, compute present value, calculate sensitivities,
and perform market risk analytics without modifying the trade objects.

Next Module

Schedule Generation

The pricing engine cannot value a swap directly.

It first needs to determine when each coupon is paid.

The next module will generate payment schedules from the trade dates and
payment frequency.